# NLP-Based Fake News Detection Using mBERT
### Multilingual Support: English, Hindi, Odia & other Indian Languages
**Domain:** NLP | **Architecture:** Multilingual BERT (mBERT) fine-tuning  
**Deployment:** Streamlit web app


## 1. Install Required Libraries

In [ ]:
# Install required packages (run once in Colab / local environment)
!pip install transformers==4.40.0 datasets torch scikit-learn pandas numpy matplotlib seaborn streamlit pyngrok --quiet


## 2. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import torch
import re
import os
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, classification_report,
                             confusion_matrix, f1_score)
from torch.utils.data import Dataset, DataLoader
from transformers import (BertTokenizer, BertForSequenceClassification,
                          AdamW, get_linear_schedule_with_warmup)
import warnings
warnings.filterwarnings('ignore')

# Device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")


## 3. Dataset Loading
### Option A – Use existing English Fake/True CSVs (upload to `/content/`)
### Option B – Use the built-in multilingual sample dataset (no upload needed)

We combine both options: English data from CSV files **+ synthetic Hindi/Odia samples**
so the model genuinely handles Indian languages.


In [ ]:
# ── Option A: load English CSVs if available ──────────────────────────────────
def load_english_csvs():
    try:
        fake = pd.read_csv('/content/Fake.csv')
        true = pd.read_csv('/content/True.csv')
        fake['label'] = 0
        true['label'] = 1
        # Keep BOTH title and text; combine for richer signal
        for df in [fake, true]:
            if 'title' in df.columns and 'text' in df.columns:
                df['content'] = df['title'].fillna('') + ' ' + df['text'].fillna('')
            elif 'text' in df.columns:
                df['content'] = df['text'].fillna('')
            elif 'title' in df.columns:
                df['content'] = df['title'].fillna('')
        combined = pd.concat([fake[['content','label']],
                              true[['content','label']]], ignore_index=True)
        print(f"Loaded English CSVs: {len(combined)} rows")
        return combined
    except FileNotFoundError:
        print("Fake.csv / True.csv not found – using multilingual sample only.")
        return None

# ── Option B: multilingual sample (English + Hindi + Odia) ────────────────────
def create_multilingual_sample():
    data = {
        'content': [
            # English – Fake (0)
            "SHOCKING: Government secretly planning to implant microchips in vaccines!",
            "Scientists CONFIRM: 5G towers spread coronavirus to control population.",
            "BREAKING: Politician caught stealing billions, media silent!",
            "Miracle cure found: This one fruit destroys cancer in 24 hours!",
            "EXPOSED: Moon landing was faked in Hollywood studio by NASA.",
            "Secret society controls all world governments through shadow banking.",
            "Doctors don't want you to know: This herb cures diabetes permanently.",
            "ALERT: Foreign agents have infiltrated every Indian government ministry.",
            # English – Real (1)
            "India's GDP grows by 6.5% in the second quarter of 2024, says RBI.",
            "Parliament passes new education bill focusing on digital literacy.",
            "Scientists develop new vaccine for dengue fever with 90% efficacy.",
            "Indian cricket team wins test series against Australia 3-1.",
            "Supreme Court upholds right to privacy as fundamental right.",
            "Odisha government launches new scheme for farmer welfare.",
            "ISRO successfully launches communication satellite into orbit.",
            "New metro line inaugurated in Bengaluru reducing travel time by 40%.",
            # Hindi – Fake (0)
            "चौंकाने वाली खबर: सरकार लोगों के खाने में जहर मिला रही है!",
            "वैज्ञानिकों ने खुलासा किया: मोबाइल टावर से निकलती है जानलेवा किरणें!",
            "ब्रेकिंग न्यूज़: नेता ने हजारों करोड़ रुपये चुराए, मीडिया चुप!",
            "चमत्कारी इलाज: यह एक फल 24 घंटे में कैंसर को नष्ट करता है!",
            "सनसनीखेज खुलासा: भारत सरकार विदेशी ताकतों के इशारे पर चल रही है!",
            "अलर्ट: पीने के पानी में मिलाया जा रहा है खतरनाक रसायन!",
            # Hindi – Real (1)
            "भारत की जीडीपी 2024 की दूसरी तिमाही में 6.5% बढ़ी: आरबीआई।",
            "संसद ने डिजिटल साक्षरता पर केंद्रित नया शिक्षा विधेयक पारित किया।",
            "इसरो ने संचार उपग्रह को सफलतापूर्वक कक्षा में स्थापित किया।",
            "ओडिशा सरकार ने किसान कल्याण के लिए नई योजना शुरू की।",
            "सुप्रीम कोर्ट ने निजता के अधिकार को मौलिक अधिकार के रूप में बरकरार रखा।",
            "भारतीय क्रिकेट टीम ने ऑस्ट्रेलिया के खिलाफ टेस्ट सीरीज 3-1 से जीती।",
            # Odia – Fake (0)
            "ଚମ୍କାଇ ଦେଉଥିବା ଖବର: ସରକାର ଲୋକଙ୍କ ଖାଦ୍ୟରେ ବିଷ ମିଶାଉଛନ୍ତି!",
            "ବୈଜ୍ଞାନିକମାନେ ପ୍ରକାଶ କଲେ: ମୋବାଇଲ ଟାୱାରରୁ ମାରାତ୍ମକ ବିକିରଣ ବାହାରୁଛି!",
            "ବ୍ରେକିଂ ନ୍ୟୁଜ: ନେତା ହଜାର ହଜାର କୋଟି ଟଙ୍କା ଚୋରି କଲେ!",
            "ଅଦ୍ଭୁତ ଖବର: ଏହି ଔଷଧ ୨୪ ଘଣ୍ଟା ମଧ୍ୟରେ କ୍ୟାନ୍ସର ଭଲ କରିଦିଏ!",
            "ସତର୍କ ରୁହନ୍ତୁ: ପାନୀୟ ଜଳରେ ବିପଜ୍ଜନକ ରାସାୟନିକ ମିଶାଯାଉଛି!",
            # Odia – Real (1)
            "ଭାରତର ଜିଡିପି ୨୦୨୪ ର ଦ୍ୱିତୀୟ ତ୍ରୟମାସରେ ୬.୫% ବୃଦ୍ଧି ପାଇଛି।",
            "ଓଡ଼ିଶା ସରକାର କୃଷକ କଲ୍ୟାଣ ପାଇଁ ନୂଆ ଯୋଜନା ଆରମ୍ଭ କଲେ।",
            "ଇସ୍ରୋ ଯୋଗାଯୋଗ ଉପଗ୍ରହକୁ ସଫଳତାର ସହ କକ୍ଷପଥରେ ସ୍ଥାପନ କଲା।",
            "ସୁପ୍ରିମ କୋର୍ଟ ଗୋପନୀୟତାର ଅଧିକାରକୁ ମୌଳିକ ଅଧିକାର ବୋଲି ସ୍ଥିର ରଖିଲା।",
            "ଭାରତୀୟ କ୍ରିକେଟ ଦଳ ଅଷ୍ଟ୍ରେଲିଆ ବିରୁଦ୍ଧ ଟେଷ୍ଟ ସିରିଜ ୩-୧ ରେ ଜିତିଲା।",
        ],
        'label': [
            0,0,0,0,0,0,0,0,  # English fake
            1,1,1,1,1,1,1,1,  # English real
            0,0,0,0,0,0,       # Hindi fake
            1,1,1,1,1,1,       # Hindi real
            0,0,0,0,0,         # Odia fake
            1,1,1,1,1,         # Odia real
        ]
    }
    df = pd.DataFrame(data)
    print(f"Multilingual sample created: {len(df)} rows")
    return df

# ── Combine ────────────────────────────────────────────────────────────────────
english_df = load_english_csvs()
multilingual_df = create_multilingual_sample()

if english_df is not None:
    # Sample to avoid heavy imbalance with large CSVs
    sample_size = min(2000, len(english_df))
    news_df = pd.concat([english_df.sample(sample_size, random_state=42),
                         multilingual_df], ignore_index=True)
else:
    news_df = multilingual_df

news_df.drop_duplicates(subset='content', inplace=True)
news_df.dropna(subset=['content'], inplace=True)
news_df.reset_index(drop=True, inplace=True)

print(f"\nFinal dataset shape: {news_df.shape}")
print(news_df['label'].value_counts().rename({0:'Fake', 1:'Real'}))


## 4. Text Preprocessing (Multilingual-Safe)

In [ ]:
def preprocess_text(text):
    """
    Multilingual-safe cleaning:
    - Keeps Devanagari (Hindi), Odia, and all Unicode letters
    - Only removes noise like URLs, extra whitespace, HTML tags
    - Does NOT use English stop-word lists (breaks Indian languages)
    """
    text = str(text)
    text = re.sub(r'http\S+|www\.\S+', '', text)       # remove URLs
    text = re.sub(r'<.*?>', '', text)                     # remove HTML
    text = re.sub(r'[^\w\s\u0900-\u097F\u0B00-\u0B7F]', ' ', text)  # keep Unicode letters
    text = re.sub(r'\s+', ' ', text).strip()
    return text

news_df['clean_content'] = news_df['content'].apply(preprocess_text)
print("Sample cleaned texts:")
for i in [0, 8, 16, 24, 30]:
    if i < len(news_df):
        lang = "EN" if i < 16 else ("HI" if i < 28 else "OR")
        print(f"  [{lang} | {'Real' if news_df.loc[i,'label']==1 else 'Fake'}] {news_df.loc[i,'clean_content'][:80]}...")


## 5. PyTorch Dataset & mBERT Tokenizer

In [ ]:
MODEL_NAME = 'bert-base-multilingual-cased'   # handles 104 languages incl. Hindi & Odia
MAX_LEN    = 128
BATCH_SIZE = 16

print(f"Loading tokenizer: {MODEL_NAME}")
tokenizer = BertTokenizer.from_pretrained(MODEL_NAME)

class FakeNewsDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len):
        self.texts     = texts.reset_index(drop=True)
        self.labels    = labels.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_len   = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.texts[idx],
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        return {
            'input_ids':      encoding['input_ids'].squeeze(0),
            'attention_mask': encoding['attention_mask'].squeeze(0),
            'labels':         torch.tensor(self.labels[idx], dtype=torch.long)
        }

# ── Split ──────────────────────────────────────────────────────────────────────
X = news_df['clean_content']
y = news_df['label']

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp)

train_ds = FakeNewsDataset(X_train, y_train, tokenizer, MAX_LEN)
val_ds   = FakeNewsDataset(X_val,   y_val,   tokenizer, MAX_LEN)
test_ds  = FakeNewsDataset(X_test,  y_test,  tokenizer, MAX_LEN)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE)

print(f"Train: {len(train_ds)} | Val: {len(val_ds)} | Test: {len(test_ds)}")


## 6. Load & Configure mBERT Model

In [ ]:
EPOCHS        = 3      # increase to 5 for better accuracy on large dataset
LEARNING_RATE = 2e-5

model = BertForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    output_attentions=False,
    output_hidden_states=False
)
model = model.to(device)

optimizer = AdamW(model.parameters(), lr=LEARNING_RATE, eps=1e-8)
total_steps = len(train_loader) * EPOCHS
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=int(0.1 * total_steps),
    num_training_steps=total_steps
)

print(model.config)
print(f"\nTotal trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")


## 7. Training Loop

In [ ]:
def train_epoch(model, loader, optimizer, scheduler, device):
    model.train()
    total_loss, correct, total = 0, 0, 0
    for batch in loader:
        optimizer.zero_grad()
        input_ids      = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels         = batch['labels'].to(device)

        outputs = model(input_ids=input_ids,
                        attention_mask=attention_mask,
                        labels=labels)
        loss    = outputs.loss
        logits  = outputs.logits

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()

        total_loss += loss.item()
        preds   = torch.argmax(logits, dim=1)
        correct += (preds == labels).sum().item()
        total   += labels.size(0)

    return total_loss / len(loader), correct / total


def eval_epoch(model, loader, device):
    model.eval()
    total_loss, correct, total = 0, 0, 0
    all_preds, all_labels = [], []
    with torch.no_grad():
        for batch in loader:
            input_ids      = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels         = batch['labels'].to(device)

            outputs = model(input_ids=input_ids,
                            attention_mask=attention_mask,
                            labels=labels)
            loss   = outputs.loss
            logits = outputs.logits

            total_loss += loss.item()
            preds   = torch.argmax(logits, dim=1)
            correct += (preds == labels).sum().item()
            total   += labels.size(0)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    return total_loss / len(loader), correct / total, all_preds, all_labels


# ── Run training ───────────────────────────────────────────────────────────────
history = {'train_loss':[], 'val_loss':[], 'train_acc':[], 'val_acc':[]}

print(f"Training mBERT for {EPOCHS} epochs on {device}...")
for epoch in range(EPOCHS):
    tr_loss, tr_acc           = train_epoch(model, train_loader, optimizer, scheduler, device)
    vl_loss, vl_acc, _, _     = eval_epoch(model, val_loader, device)

    history['train_loss'].append(tr_loss)
    history['val_loss'].append(vl_loss)
    history['train_acc'].append(tr_acc)
    history['val_acc'].append(vl_acc)

    print(f"Epoch {epoch+1}/{EPOCHS}  |  "
          f"Train Loss: {tr_loss:.4f}  Acc: {tr_acc:.4f}  |  "
          f"Val Loss: {vl_loss:.4f}  Acc: {vl_acc:.4f}")

print("Training complete.")


## 8. Evaluation on Test Set

In [ ]:
_, test_acc, y_pred, y_true = eval_epoch(model, test_loader, device)

print(f"Test Accuracy: {test_acc:.4f}")
print(f"F1 Score (weighted): {f1_score(y_true, y_pred, average='weighted'):.4f}")
print("\nClassification Report:")
print(classification_report(y_true, y_pred, target_names=['Fake', 'Real']))


## 9. Visualizations

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Loss curve
axes[0].plot(history['train_loss'], marker='o', label='Train Loss')
axes[0].plot(history['val_loss'],   marker='s', label='Val Loss')
axes[0].set_title('Loss Curve'); axes[0].set_xlabel('Epoch')
axes[0].legend(); axes[0].grid(True)

# Accuracy curve
axes[1].plot(history['train_acc'], marker='o', label='Train Acc')
axes[1].plot(history['val_acc'],   marker='s', label='Val Acc')
axes[1].set_title('Accuracy Curve'); axes[1].set_xlabel('Epoch')
axes[1].legend(); axes[1].grid(True)

# Confusion matrix
cm = confusion_matrix(y_true, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[2],
            xticklabels=['Fake','Real'], yticklabels=['Fake','Real'])
axes[2].set_xlabel('Predicted'); axes[2].set_ylabel('True')
axes[2].set_title('Confusion Matrix')

plt.tight_layout()
plt.savefig('evaluation_plots.png', dpi=150)
plt.show()
print("Plot saved as evaluation_plots.png")


## 10. Save Model & Tokenizer

In [ ]:
SAVE_PATH = '/content/fake_news_mbert_model'
os.makedirs(SAVE_PATH, exist_ok=True)

model.save_pretrained(SAVE_PATH)
tokenizer.save_pretrained(SAVE_PATH)
print(f"Model and tokenizer saved to: {SAVE_PATH}")


## 11. Prediction Function (English / Hindi / Odia)

In [ ]:
def predict_news(text, model=model, tokenizer=tokenizer,
                 device=device, max_len=MAX_LEN):
    """
    Predict whether a news headline/article is Fake or Real.
    Works for English, Hindi (Devanagari), and Odia.
    """
    model.eval()
    clean = preprocess_text(text)
    encoding = tokenizer(
        clean,
        max_length=max_len,
        padding='max_length',
        truncation=True,
        return_tensors='pt'
    )
    input_ids      = encoding['input_ids'].to(device)
    attention_mask = encoding['attention_mask'].to(device)

    with torch.no_grad():
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        probs   = torch.softmax(outputs.logits, dim=1).cpu().numpy()[0]
        pred    = int(torch.argmax(outputs.logits, dim=1))

    label = 'REAL ✅' if pred == 1 else 'FAKE ❌'
    return {
        'prediction': label,
        'confidence': f"{probs[pred]*100:.1f}%",
        'fake_prob':  f"{probs[0]*100:.1f}%",
        'real_prob':  f"{probs[1]*100:.1f}%"
    }

# ── Test on sample headlines ───────────────────────────────────────────────────
test_headlines = [
    # English
    "Scientists discover water on Mars surface – NASA confirms.",
    "SHOCKING: Government adds poison to tap water to control population!",
    # Hindi
    "भारत की जीडीपी 2024 में 6.5% बढ़ी: आरबीआई की रिपोर्ट।",
    "चौंकाने वाली खबर: सरकार लोगों के खाने में जहर मिला रही है!",
    # Odia
    "ଓଡ଼ିଶା ସରକାର କୃଷକ କଲ୍ୟାଣ ପାଇଁ ନୂଆ ଯୋଜନା ଆରମ୍ଭ କଲେ।",
    "ଚମ୍କାଇ ଦେଉଥିବା ଖବର: ସରକାର ଲୋକଙ୍କ ଖାଦ୍ୟରେ ବିଷ ମିଶାଉଛନ୍ତି!",
]

print("=" * 70)
print(f"{'Headline':<55} {'Result':<12} {'Confidence'}")
print("=" * 70)
for h in test_headlines:
    result = predict_news(h)
    print(f"{h[:52]:<55} {result['prediction']:<12} {result['confidence']}")


## 12. Streamlit Web App (Deploy Locally or on Colab)

In [ ]:
# ── Write app.py ────────────────────────────────────────────────────────────
app_code = '''
import streamlit as st
import torch
import re
from transformers import BertTokenizer, BertForSequenceClassification

MODEL_PATH = "./fake_news_mbert_model"   # adjust path as needed
MAX_LEN    = 128

@st.cache_resource
def load_model():
    tok   = BertTokenizer.from_pretrained(MODEL_PATH)
    model = BertForSequenceClassification.from_pretrained(MODEL_PATH)
    model.eval()
    return model, tok

def preprocess(text):
    text = re.sub(r"http\\S+|www\\.\\S+", "", text)
    text = re.sub(r"<.*?>", "", text)
    text = re.sub(r"[^\\w\\s\\u0900-\\u097F\\u0B00-\\u0B7F]", " ", text)
    return re.sub(r"\\s+", " ", text).strip()

def predict(text, model, tokenizer):
    clean = preprocess(text)
    enc   = tokenizer(clean, max_length=MAX_LEN, padding="max_length",
                      truncation=True, return_tensors="pt")
    with torch.no_grad():
        logits = model(**enc).logits
        probs  = torch.softmax(logits, dim=1)[0].tolist()
        pred   = int(torch.argmax(logits))
    return pred, probs

# ── UI ───────────────────────────────────────────────────────────────────────
st.set_page_config(page_title="Fake News Detector", page_icon="📰", layout="centered")

st.title("📰 Multilingual Fake News Detector")
st.markdown("Supports **English**, **Hindi (हिंदी)**, and **Odia (ଓଡ଼ିଆ)** news headlines.")

news_input = st.text_area("Paste a news headline or article:", height=120,
                           placeholder="Enter English, Hindi, or Odia text here...")

if st.button("🔍 Analyze"):
    if news_input.strip():
        with st.spinner("Analyzing..."):
            model, tokenizer = load_model()
            pred, probs = predict(news_input, model, tokenizer)

        if pred == 1:
            st.success(f"✅ **REAL NEWS** — Confidence: {probs[1]*100:.1f}%")
        else:
            st.error(f"❌ **FAKE NEWS** — Confidence: {probs[0]*100:.1f}%")

        col1, col2 = st.columns(2)
        col1.metric("Fake Probability",  f"{probs[0]*100:.1f}%")
        col2.metric("Real Probability",  f"{probs[1]*100:.1f}%")
    else:
        st.warning("Please enter some text first.")

st.markdown("---")
st.caption("Model: bert-base-multilingual-cased | Fine-tuned for fake news detection")
'''

with open('/content/app.py', 'w', encoding='utf-8') as f:
    f.write(app_code)
print("app.py written to /content/app.py")


In [ ]:
# ── Launch Streamlit via pyngrok (run this cell LAST) ──────────────────────
# Uncomment to run in Google Colab:

# from pyngrok import ngrok
# import subprocess, time

# proc = subprocess.Popen(['streamlit', 'run', '/content/app.py',
#                          '--server.port=8501', '--server.headless=true'])
# time.sleep(5)
# public_url = ngrok.connect(8501)
# print(f"\n🌐 Streamlit App URL: {public_url}")

# ── Or run locally ──────────────────────────────────────────────────────────
# 1. Copy the saved model folder to the same directory as app.py
# 2. Run:  streamlit run app.py

print("To launch locally:")
print("  1. Save model: model.save_pretrained('./fake_news_mbert_model')")
print("  2. Run: streamlit run app.py")


## Summary

| Component | This Notebook |
|---|---|
| **Architecture** | mBERT (`bert-base-multilingual-cased`) |
| **Languages** | English, Hindi (हिंदी), Odia (ଓଡ଼ିଆ) |
| **Input** | Headlines + Article text (combined) |
| **Preprocessing** | Unicode-safe (keeps Devanagari & Odia scripts) |
| **Training** | Fine-tuned with AdamW + linear warmup scheduler |
| **Evaluation** | Accuracy, F1, Classification Report, Confusion Matrix |
| **Model Saving** | `save_pretrained()` for easy reload |
| **Deployment** | Streamlit web app (`app.py`) |

### Next Steps (optional upgrades)
- Collect a larger Hindi/Odia fake news dataset (e.g., from fact-checking sites like Boom, Alt News)
- Increase `EPOCHS` to 5 and use a GPU runtime for better accuracy
- Package the Streamlit app as a **Chrome extension** using a Flask API backend
